# REPORT Notebook 4 — Évaluation & Prédictions
**ISIC Project | Segmentation de Lésions Cutanées**

Ce notebook couvre :
- OK Chargement du meilleur modèle
- OK Évaluation sur le jeu de test
- OK Métriques : Dice, IoU, Précision, Recall
- OK Visualisation des prédictions
- OK Export des résultats

## LINK Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_PATH  = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH   = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Masques')
SRC_PATH      = os.path.join(PROJECT_PATH, 'src')
OUTPUTS_PATH  = os.path.join(PROJECT_PATH, 'outputs')
MODEL_PATH    = os.path.join(OUTPUTS_PATH, 'best_unet_isic.pth')
sys.path.append(SRC_PATH)

!pip install -q albumentations opencv-python-headless scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'OK Setup OK | Device : {device}')

## MAGNIFYING_GLASS Chargement des fichiers (filtre images uniquement)

In [ ]:
# OK CORRECTION : filtre les fichiers non-images (ATTRIBUTION.txt, etc.)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])

assert len(all_images) == len(all_masks), \
    f'CROSS_MARK Nombre images ({len(all_images)}) != masques ({len(all_masks)})'
print(f'OK {len(all_images)} paires chargées')

## PACKAGE Dataset Test & Modèle

In [ ]:
IMG_SIZE = 256

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask'].unsqueeze(0)
        return img.float(), mask.float()

# U-Net
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2,2)
        for f in features:
            self.downs.append(DoubleConv(in_channels, f)); in_channels = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape: x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = self.ups[i+1](torch.cat([skip, x], dim=1))
        return self.final(x)

# Split identique au notebook 03
_, temp_imgs, _, temp_msks = train_test_split(all_images, all_masks, test_size=0.3, random_state=42)
_, test_imgs, _, test_msks = train_test_split(temp_imgs, temp_msks, test_size=0.5, random_state=42)

test_loader = DataLoader(
    ISICDataset(test_imgs, test_msks, val_transform),
    batch_size=8, shuffle=False, num_workers=2)

# Chargement du modèle
model = UNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print(f'OK Modèle chargé depuis : {MODEL_PATH}')
print(f'OK Test set : {len(test_imgs)} images | {len(test_loader)} batchs')

## REPORT Calcul des métriques sur le test set

In [ ]:
def compute_metrics(pred, target, threshold=0.5, smooth=1e-6):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    tp = (pred_bin * target).sum(dim=(2,3))
    fp = (pred_bin * (1-target)).sum(dim=(2,3))
    fn = ((1-pred_bin) * target).sum(dim=(2,3))
    dice      = ((2*tp + smooth) / (2*tp + fp + fn + smooth)).mean().item()
    iou       = ((tp + smooth) / (tp + fp + fn + smooth)).mean().item()
    precision = ((tp + smooth) / (tp + fp + smooth)).mean().item()
    recall    = ((tp + smooth) / (tp + fn + smooth)).mean().item()
    return {'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall}

all_metrics = {'dice': [], 'iou': [], 'precision': [], 'recall': []}

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Évaluation'):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        m = compute_metrics(preds, masks)
        for k in all_metrics: all_metrics[k].append(m[k])

results = {k: float(np.mean(v)) for k, v in all_metrics.items()}

print('\nREPORT Résultats sur le Test Set :')
print(f'   Dice Score : {results["dice"]:.4f}')
print(f'   IoU        : {results["iou"]:.4f}')
print(f'   Précision  : {results["precision"]:.4f}')
print(f'   Recall     : {results["recall"]:.4f}')

# Sauvegarde
os.makedirs(OUTPUTS_PATH, exist_ok=True)
with open(os.path.join(OUTPUTS_PATH, 'test_metrics.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('\nSAVE Métriques sauvegardées : outputs/test_metrics.json')

## PALETTE Visualisation des prédictions

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

imgs, masks = next(iter(test_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(imgs.to(device))).cpu()

n = 4
fig, axes = plt.subplots(4, n, figsize=(4*n, 16))
for i in range(n):
    img_show  = (imgs[i] * std + mean).clamp(0,1).permute(1,2,0).numpy()
    mask_show = masks[i, 0].numpy()
    pred_show = (preds[i, 0] > 0.5).float().numpy()
    overlay   = (img_show * 255).astype(np.uint8).copy()
    overlay[pred_show > 0] = [255, 80, 80]

    axes[0,i].imshow(img_show);                 axes[0,i].set_title('Image');       axes[0,i].axis('off')
    axes[1,i].imshow(mask_show, cmap='gray');   axes[1,i].set_title('Vrai masque'); axes[1,i].axis('off')
    axes[2,i].imshow(pred_show, cmap='gray');   axes[2,i].set_title('Prédiction');  axes[2,i].axis('off')
    axes[3,i].imshow(overlay);                  axes[3,i].set_title('Overlay');     axes[3,i].axis('off')

plt.suptitle('Résultats de segmentation ISIC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'predictions_visualization.png'), dpi=150)
plt.show()
print('SAVE Sauvegardé : outputs/predictions_visualization.png')

## CHART_UP Rapport final — graphique des métriques

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
names  = ['Dice Score', 'IoU', 'Précision', 'Recall']
values = [results['dice'], results['iou'], results['precision'], results['recall']]
colors = ['#0E7C7B', '#14B8A6', '#F26419', '#334155']

bars = ax.barh(names, values, color=colors, edgecolor='white', height=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 1.1)
ax.set_xlabel('Score', fontsize=12)
ax.set_title('REPORT Métriques finales — ISIC Segmentation (Test Set)', fontsize=13, fontweight='bold')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='Seuil 0.8')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'final_metrics.png'), dpi=150)
plt.show()

print('\n🎉 Évaluation terminée !')
print('\nFOLDER Fichiers générés dans outputs/ :')
for f in sorted(os.listdir(OUTPUTS_PATH)):
    print(f'   FILE {f}')

## OK Résumé final
- Modèle évalué sur le test set
- Métriques sauvegardées : `outputs/test_metrics.json`
- Visualisations : `outputs/predictions_visualization.png`
- Rapport : `outputs/final_metrics.png`

🎉 **Projet ISIC complet !**